# Graph Property Analysis

Counts graph edges/distances/properties for saved sanity, progressive-noise, topology, and density DAGs, then joins them with predictive metrics and intervention AUC when experiment results are available.

In [ ]:
from pathlib import Path
from collections import deque
import csv
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 160)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASETS = {
    'celeba': {
        'display': 'CelebA',
        'model': 'Standard_CelebA',
        'base_dag': PROJECT_ROOT / 'data/CelebA/final_DAG_unfair.csv',
        'data_root': PROJECT_ROOT / 'data/CelebA',
        'num_classes': 1,
    },
    'cub': {
        'display': 'CUB',
        'model': 'Standard_CUB',
        'base_dag': PROJECT_ROOT / 'data/CUB/CUB_DAG_only_Gc.csv',
        'data_root': PROJECT_ROOT / 'data/CUB',
        'num_classes': 200,
    },
    'cfmnist': {
        'display': 'Complete_Concept_FMNIST',
        'model': 'Standard_FashionMNIST',
        'base_dag': PROJECT_ROOT / 'data/FashionMNIST/Complete_Concept_FMNIST_DAG.csv',
        'data_root': PROJECT_ROOT / 'data/FashionMNIST',
        'num_classes': 10,
    },
}

FAMILIES = {
    'graph_sanity': 'graph_sanity',
    'progressive_noise': 'progressive_noise_graph',
    'topology_random_full': 'topology_random_full',
    'density_random_full': 'density_random_full',
}


## Helper Functions

In [ ]:
def read_bool_dag(path):
    with open(path, newline='') as f:
        rows = list(csv.reader(f))
    names = rows[0][1:]
    matrix = np.array([[cell == 'True' for cell in row[1:]] for row in rows[1:]], dtype=bool)
    return names, matrix

def variant_from_path(path, family):
    stem = path.stem
    if family == 'graph_sanity':
        for variant in ['random_same_edges', 'all_ones', 'identity', 'original']:
            if stem.endswith('_' + variant):
                return variant
    if family == 'progressive_noise':
        match = re.search(r'noise_(\d+)', stem)
        return f"noise_{match.group(1)}" if match else stem
    if family == 'topology_random_full':
        match = re.search(r'(rand_topology_\d+)', stem)
        return match.group(1) if match else stem
    if family == 'density_random_full':
        match = re.search(r'(edges_\d+)', stem)
        return match.group(1) if match else stem
    return stem

def shortest_path_stats(adj):
    n = adj.shape[0]
    lengths = []
    reach = np.eye(n, dtype=bool) | adj.copy()
    for src in range(n):
        dist = [-1] * n
        dist[src] = 0
        queue = deque([src])
        while queue:
            node = queue.popleft()
            for dst in np.flatnonzero(adj[node]):
                if dist[dst] < 0:
                    dist[dst] = dist[node] + 1
                    queue.append(dst)
        lengths.extend([d for d in dist if d > 0])
        reach[src] = np.array([d >= 0 for d in dist], dtype=bool)
    return reach, (float(np.mean(lengths)) if lengths else np.nan)

def weak_component_stats(adj):
    undirected = adj | adj.T
    n = adj.shape[0]
    seen = np.zeros(n, dtype=bool)
    sizes = []
    for start in range(n):
        if seen[start]:
            continue
        queue = deque([start])
        seen[start] = True
        size = 0
        while queue:
            node = queue.popleft()
            size += 1
            for dst in np.flatnonzero(undirected[node]):
                if not seen[dst]:
                    seen[dst] = True
                    queue.append(dst)
        sizes.append(size)
    return len(sizes), max(sizes) / n if sizes else np.nan

def graph_properties(dataset, family, variant, path, names, adj, original, num_classes):
    n = adj.shape[0]
    k = n - num_classes
    task_nodes = list(range(k, n))
    concept_nodes = list(range(k))
    total_positions = n * n
    edges = int(adj.sum())
    original_edges = int(original.sum())
    diff = adj != original
    added = int((adj & ~original).sum())
    deleted = int((~adj & original).sum())
    out_degree = adj.sum(axis=1)
    in_degree = adj.sum(axis=0)
    total_degree = out_degree + in_degree
    reach, avg_path = shortest_path_stats(adj)
    n_components, largest_component_frac = weak_component_stats(adj)
    if concept_nodes and task_nodes:
        concept_to_task = reach[np.ix_(concept_nodes, task_nodes)]
        task_reachability = float(concept_to_task.any(axis=1).mean())
        avg_task_targets_reached = float(concept_to_task.sum(axis=1).mean())
    else:
        task_reachability = np.nan
        avg_task_targets_reached = np.nan
    if k > 1:
        concept_reach = reach[:k, :k].copy()
        np.fill_diagonal(concept_reach, False)
        concept_pair_reachability = float(concept_reach.sum() / (k * (k - 1)))
    else:
        concept_pair_reachability = np.nan
    return {
        'dataset': dataset,
        'family': family,
        'variant': variant,
        'dag_path': str(path),
        'num_nodes': n,
        'num_concepts': k,
        'num_task_nodes': num_classes,
        'edge_count': edges,
        'original_edge_count': original_edges,
        'edge_delta': edges - original_edges,
        'density': edges / total_positions,
        'total_positions': total_positions,
        'hamming_distance': int(diff.sum()),
        'graph_distance': float(diff.sum() / total_positions),
        'added_edges': added,
        'deleted_edges': deleted,
        'self_edges': int(np.trace(adj)),
        'concept_concept_edges': int(adj[:k, :k].sum()),
        'concept_task_edges': int(adj[:k, k:].sum()) if num_classes else 0,
        'task_concept_edges': int(adj[k:, :k].sum()) if num_classes else 0,
        'task_task_edges': int(adj[k:, k:].sum()) if num_classes else 0,
        'avg_out_degree': float(out_degree.mean()),
        'avg_in_degree': float(in_degree.mean()),
        'degree_variance': float(total_degree.var()),
        'max_out_degree': int(out_degree.max()) if n else 0,
        'max_in_degree': int(in_degree.max()) if n else 0,
        'weak_components': n_components,
        'largest_weak_component_frac': largest_component_frac,
        'avg_shortest_path_directed_reachable': avg_path,
        'task_reachability': task_reachability,
        'avg_task_targets_reached_per_concept': avg_task_targets_reached,
        'concept_pair_reachability': concept_pair_reachability,
    }


## Graph Property Table

In [ ]:
rows = []
for dataset, meta in DATASETS.items():
    if not meta['base_dag'].exists():
        continue
    names, original = read_bool_dag(meta['base_dag'])
    for family, subdir in FAMILIES.items():
        dag_dir = meta['data_root'] / subdir
        if not dag_dir.exists():
            continue
        for path in sorted(dag_dir.glob('*.csv')):
            if 'metadata' in path.name.lower():
                continue
            graph_names, adj = read_bool_dag(path)
            if graph_names != names:
                print(f'Skipping {path}: node labels differ from base DAG')
                continue
            variant = variant_from_path(path, family)
            rows.append(graph_properties(dataset, family, variant, path, graph_names, adj, original, meta['num_classes']))

graph_props = pd.DataFrame(rows).sort_values(['dataset', 'family', 'variant']) if rows else pd.DataFrame()
display(graph_props)


## Load Predictive Metrics and Intervention AUC

In [ ]:
def experiment_root(dataset, family):
    meta = DATASETS[dataset]
    return PROJECT_ROOT / 'experiments' / meta['display'] / 'train_cbm' / meta['model'] / f'{dataset}_{family}' / f'CREAM_{dataset}_{family}'

def load_last_metrics():
    rows = []
    for dataset in DATASETS:
        for family in FAMILIES:
            root = experiment_root(dataset, family)
            if not root.exists():
                continue
            for csv_path in sorted(root.glob('*/last_metrics/*.csv')):
                df = pd.read_csv(csv_path)
                if df.empty:
                    continue
                row = df.iloc[0].to_dict()
                row['dataset'] = dataset
                row['family'] = family
                row['variant'] = csv_path.relative_to(root).parts[0]
                row['metrics_path'] = str(csv_path)
                rows.append(row)
    return pd.DataFrame(rows)

def intervention_accuracy_column(df):
    for col in ['test_task_accuracy', 'task_accuracy', 'accuracy', 'test_acc']:
        if col in df.columns:
            return col
    matches = [col for col in df.columns if 'accuracy' in col.lower() or col.lower().endswith('_acc')]
    return matches[0] if matches else None

def load_intervention_auc():
    rows = []
    for dataset in DATASETS:
        for family in FAMILIES:
            root = experiment_root(dataset, family)
            if not root.exists():
                continue
            for csv_path in sorted(root.glob('*/lightning_logs/**/intervention_results.csv')):
                df = pd.read_csv(csv_path)
                if df.empty or 'num_interventions' not in df.columns:
                    continue
                acc_col = intervention_accuracy_column(df)
                if acc_col is None:
                    continue
                variant = csv_path.relative_to(root).parts[0]
                group_col = 'group_interventions' if 'group_interventions' in df.columns else None
                groups = df.groupby(group_col) if group_col else [(None, df)]
                for group_value, group_df in groups:
                    group_df = group_df.dropna(subset=['num_interventions', acc_col]).sort_values('num_interventions')
                    if group_df.empty:
                        continue
                    x = group_df['num_interventions'].to_numpy(dtype=float)
                    y = group_df[acc_col].to_numpy(dtype=float)
                    x_norm = x / x.max() if x.max() > 0 else x
                    auc = float(np.trapz(y, x_norm)) if len(x_norm) > 1 else float(y[0])
                    suffix = 'all' if group_value is None else str(group_value).lower()
                    rows.append({
                        'dataset': dataset,
                        'family': family,
                        'variant': variant,
                        'group_interventions': group_value,
                        f'intervention_auc_{suffix}': auc,
                        f'intervention_points_{suffix}': len(group_df),
                        'intervention_path': str(csv_path),
                    })
    if not rows:
        return pd.DataFrame()
    raw = pd.DataFrame(rows)
    value_cols = [c for c in raw.columns if c.startswith('intervention_auc_') or c.startswith('intervention_points_')]
    return raw.groupby(['dataset', 'family', 'variant'], as_index=False)[value_cols].first()

metrics = load_last_metrics()
intervention_auc = load_intervention_auc()
print(f'Loaded metric rows: {len(metrics)}')
print(f'Loaded intervention-AUC rows: {len(intervention_auc)}')
display(metrics.head())
display(intervention_auc.head())


## Combined Table

In [ ]:
combined = graph_props.copy()
if not metrics.empty:
    combined = combined.merge(metrics, on=['dataset', 'family', 'variant'], how='left')
if not intervention_auc.empty:
    combined = combined.merge(intervention_auc, on=['dataset', 'family', 'variant'], how='left')

key_cols = [
    'dataset', 'family', 'variant', 'edge_count', 'density', 'graph_distance',
    'added_edges', 'deleted_edges', 'avg_out_degree', 'degree_variance',
    'weak_components', 'largest_weak_component_frac', 'avg_shortest_path_directed_reachable',
    'task_reachability', 'concept_pair_reachability',
    'test_task_accuracy', 'test_concept_accuracy', 'CCI', 'PFI_concept_importance', 'PFI_side_importance',
    'intervention_auc_false', 'intervention_auc_true',
]
display(combined[[c for c in key_cols if c in combined.columns]])


## Correlations

In [ ]:
property_cols = [
    'edge_count', 'density', 'graph_distance', 'added_edges', 'deleted_edges',
    'self_edges', 'concept_concept_edges', 'concept_task_edges', 'task_concept_edges',
    'avg_out_degree', 'avg_in_degree', 'degree_variance', 'max_out_degree', 'max_in_degree',
    'weak_components', 'largest_weak_component_frac', 'avg_shortest_path_directed_reachable',
    'task_reachability', 'avg_task_targets_reached_per_concept', 'concept_pair_reachability',
]
outcome_cols = [
    'test_task_accuracy', 'test_concept_accuracy', 'CCI', 'PFI_concept_importance', 'PFI_side_importance',
    'intervention_auc_false', 'intervention_auc_true', 'intervention_auc_all',
]

corr_rows = []
for dataset, dataset_df in combined.groupby('dataset'):
    for family, df in dataset_df.groupby('family'):
        for outcome in outcome_cols:
            if outcome not in df.columns or df[outcome].notna().sum() < 3:
                continue
            for prop in property_cols:
                if prop not in df.columns or df[prop].notna().sum() < 3:
                    continue
                valid = df[[prop, outcome]].dropna()
                if len(valid) < 3 or valid[prop].nunique() < 2 or valid[outcome].nunique() < 2:
                    continue
                corr_rows.append({
                    'dataset': dataset,
                    'family': family,
                    'property': prop,
                    'outcome': outcome,
                    'pearson_corr': valid[prop].corr(valid[outcome]),
                    'n': len(valid),
                })
correlations = pd.DataFrame(corr_rows)
if correlations.empty:
    print('No correlations yet. Need at least 3 completed runs with varying values.')
else:
    display(correlations.sort_values('pearson_corr', key=lambda s: s.abs(), ascending=False).head(40))


## Diagnostic Plots

In [ ]:
def scatter_if_available(df, x, y, title):
    if x not in df.columns or y not in df.columns:
        print(f'Skipping {title}: missing {x} or {y}')
        return
    plot_df = df[[x, y, 'dataset', 'family', 'variant']].dropna()
    if plot_df.empty:
        print(f'Skipping {title}: no completed rows')
        return
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for (dataset, family), group in plot_df.groupby(['dataset', 'family']):
        ax.scatter(group[x], group[y], label=f'{dataset}:{family}', s=55)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(title)
    ax.grid(alpha=0.25)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

scatter_if_available(combined, 'density', 'test_task_accuracy', 'Density vs task accuracy')
scatter_if_available(combined, 'graph_distance', 'test_task_accuracy', 'Graph distance from original vs task accuracy')
scatter_if_available(combined, 'task_reachability', 'intervention_auc_false', 'Task reachability vs individual intervention AUC')
scatter_if_available(combined, 'concept_pair_reachability', 'intervention_auc_false', 'Concept reachability vs individual intervention AUC')


## Observational-Interventional Gap

In [ ]:
def gap_table(df, outcome_pred='test_task_accuracy', outcome_int='intervention_auc_false'):
    if outcome_pred not in df.columns or outcome_int not in df.columns:
        return pd.DataFrame()
    rows = []
    for (dataset, family), group in df.dropna(subset=[outcome_pred, outcome_int]).groupby(['dataset', 'family']):
        records = group.to_dict('records')
        for i in range(len(records)):
            for j in range(i + 1, len(records)):
                a, b = records[i], records[j]
                rows.append({
                    'dataset': dataset,
                    'family': family,
                    'variant_i': a['variant'],
                    'variant_j': b['variant'],
                    'delta_pred': abs(a[outcome_pred] - b[outcome_pred]),
                    'delta_int': abs(a[outcome_int] - b[outcome_int]),
                    'edge_delta_pair': abs(a['edge_count'] - b['edge_count']),
                    'graph_distance_pair': abs(a['graph_distance'] - b['graph_distance']),
                })
    return pd.DataFrame(rows)

gaps = gap_table(combined)
if gaps.empty:
    print('No gap table yet. Need task accuracy and intervention AUC results.')
else:
    display(gaps.sort_values(['delta_pred', 'delta_int'], ascending=[True, False]).head(30))
    fig, ax = plt.subplots(figsize=(6, 5))
    for (dataset, family), group in gaps.groupby(['dataset', 'family']):
        ax.scatter(group['delta_pred'], group['delta_int'], label=f'{dataset}:{family}', s=45)
    ax.set_xlabel('Delta prediction accuracy')
    ax.set_ylabel('Delta intervention AUC')
    ax.set_title('Observational-interventional gap')
    ax.grid(alpha=0.25)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
